# CellMSA demo

Prepare related-cell context, generate embeddings, and explore UMAP and scIB metrics on the supplied blood dataset. The demo uses known cell-type and donor annotations.

Run `git lfs pull` from the repository root to download the checkpoint and data. Generated files are saved under `output/notebook/`; use a fresh output directory when rerunning.


In [ ]:
from pathlib import Path
import os

ROOT = Path.cwd()
if not (ROOT / "embedding.py").is_file():
    ROOT = ROOT.parent
if not (ROOT / "embedding.py").is_file():
    raise RuntimeError("Launch Jupyter from the repository root or demo directory.")
DATA_DIR = ROOT / "demo" / "data"
SOURCE_H5AD = DATA_DIR / "blood_subset.h5ad"
RUN_DIR = ROOT / "output" / "notebook"
H5AD_PATH = RUN_DIR / "blood_subset_context.h5ad"
EMBEDDING_SCRIPT = ROOT / "embedding.sh"
EMBEDDING_PREFIX = RUN_DIR / "blood_subset_cellmsa"


In [ ]:
import sys
import random
import subprocess

import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

## 1. Prepare context

Use the original demo sampling rules (16/16/8 neighbors), with seed 0. This requires known `cell_type`, `donor_id`, and `supertype` annotations and writes a separate prepared file under `output/`. It is not a label-free test-time retrieval procedure.


In [ ]:
subprocess.run([
    sys.executable, str(ROOT / "demo/prepare_context.py"),
    "--input", str(SOURCE_H5AD), "--output", str(H5AD_PATH), "--seed", "0"
], cwd=ROOT, check=True)


## 2. Run CellMSA Embedding


In [ ]:
env = os.environ.copy()
env.update({"QUERY_H5AD": str(H5AD_PATH), "MSA_H5AD": str(H5AD_PATH),
            "OUTPUT_PREFIX": str(EMBEDDING_PREFIX), "PYTHON": sys.executable})
subprocess.run(["bash", str(EMBEDDING_SCRIPT)], cwd=ROOT, env=env, check=True)


## 3. Load Data and CellMSA Embeddings


In [ ]:
adata = sc.read_h5ad(H5AD_PATH)
embedding_paths = sorted(
    EMBEDDING_PREFIX.parent.glob(f"{EMBEDDING_PREFIX.name}_embeddings_*-*.npy"),
    key=lambda p: int(p.stem.rsplit("_", 1)[-1].split("-")[0]),
)
if not embedding_paths:
    raise FileNotFoundError("No generated embedding chunks found.")
expected_start = 0
for p in embedding_paths:
    start, end = map(int, p.stem.rsplit("_", 1)[-1].split("-"))
    if start != expected_start or end <= start:
        raise ValueError("Noncontiguous/overlapping output chunks; use a clean output prefix.")
    if np.load(p, mmap_mode="r").shape[0] != end - start:
        raise ValueError(f"Row range does not match {p.name}")
    expected_start = end
cellmsa_embeddings = np.concatenate([np.load(p) for p in embedding_paths], axis=0)

if cellmsa_embeddings.shape[0] != adata.n_obs:
    raise ValueError(
        f"Embedding rows ({cellmsa_embeddings.shape[0]:,}) do not match AnnData cells ({adata.n_obs:,})."
    )

print(f"AnnData shape: {adata.n_obs:,} cells x {adata.n_vars:,} genes")
print(f"CellMSA embedding shape: {cellmsa_embeddings.shape}")
print(f"Embedding dtype: {cellmsa_embeddings.dtype}")

display(adata.obs[[k for k in ["cell_type", "supertype", "donor_id", "assay"] if k in adata.obs]].head())

## 4. Quick Summary


In [ ]:
emb = cellmsa_embeddings.astype(np.float32, copy=False)

summary = pd.DataFrame(
    {
        "n_cells": [emb.shape[0]],
        "embedding_dim": [emb.shape[1]]
    }
)
display(summary)

display(
    adata.obs["cell_type"]
    .value_counts()
    .rename_axis("cell_type")
    .reset_index(name="n_cells")
    .head(15)
)

## 5. Visualize CellMSA Embedding Space


In [ ]:
emb_adata = sc.AnnData(X=emb)
emb_adata.obs = adata.obs.copy()

sc.pp.neighbors(emb_adata, n_neighbors=15, use_rep="X")
sc.tl.umap(emb_adata, random_state=SEED)

In [ ]:
sc.pl.umap(
    emb_adata,
    color=["cell_type", "donor_id"],
    ncols=1,
    wspace=0.3,
)

## 6. scIB Evaluation


This section evaluates the CellMSA embedding with a small scIB panel. Here `cell_type` is used as the biological label and `donor_id` is used as the batch key. Higher values indicate better performance for the metrics reported below.


In [ ]:
import warnings
try:
    from scib.metrics import ari, graph_connectivity, nmi, silhouette, silhouette_batch
except ImportError as exc:
    raise ImportError(
        "scIB is required for this evaluation section. Install scib before running this cell."
    ) from exc

SCIB_LABEL_KEY = "cell_type"
SCIB_BATCH_KEY = "donor_id"
SCIB_EMBED_KEY = "X_cellmsa"
SCIB_CLUSTER_KEY = "cellmsa_leiden"

missing_scib_cols = [
    col for col in [SCIB_LABEL_KEY, SCIB_BATCH_KEY] if col not in adata.obs
]
if missing_scib_cols:
    raise KeyError(f"Missing required scIB obs columns: {missing_scib_cols}")

adata_scib = sc.AnnData(X=emb, obs=adata.obs.copy())
adata_scib.obsm[SCIB_EMBED_KEY] = emb

sc.pp.neighbors(adata_scib, n_neighbors=15, use_rep=SCIB_EMBED_KEY)
sc.tl.leiden(
    adata_scib,
    resolution=0.1,
    key_added=SCIB_CLUSTER_KEY,
    random_state=SEED,
)

with warnings.catch_warnings():
    warnings.filterwarnings(
        "ignore",
        message="pandas.value_counts is deprecated.*",
        category=FutureWarning,
        module="scib.metrics.graph_connectivity",
    )
    scib_scores = {
        "label_ASW": silhouette(
            adata_scib,
            label_key=SCIB_LABEL_KEY,
            embed=SCIB_EMBED_KEY,
            metric="euclidean",
            scale=True,
        ),
        "batch_ASW": silhouette_batch(
            adata_scib,
            batch_key=SCIB_BATCH_KEY,
            label_key=SCIB_LABEL_KEY,
            embed=SCIB_EMBED_KEY,
            metric="euclidean",
            scale=True,
            verbose=False,
        ),
        "graph_connectivity": graph_connectivity(
            adata_scib,
            label_key=SCIB_LABEL_KEY,
        ),
        "NMI_leiden_cell_type": nmi(
            adata_scib,
            cluster_key=SCIB_CLUSTER_KEY,
            label_key=SCIB_LABEL_KEY,
        ),
        "ARI_leiden_cell_type": ari(
            adata_scib,
            cluster_key=SCIB_CLUSTER_KEY,
            label_key=SCIB_LABEL_KEY,
        ),
    }

scib_results = (
    pd.Series(scib_scores, name="score")
    .rename_axis("metric")
    .reset_index()
)
display(scib_results)

print(f"scIB label key: {SCIB_LABEL_KEY}")
print(f"scIB batch key: {SCIB_BATCH_KEY}")
print(f"Leiden clusters: {adata_scib.obs[SCIB_CLUSTER_KEY].nunique()}")